In [1]:
import pandas as pd
import geopandas as gpd
from dash import Dash, dcc, html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import plotly.express as px

# Load the Excel file
file_path = 'Copy of HEAT_Tables_20.06.24_CPedits.xlsx'
xlsx_file = pd.read_excel(file_path, sheet_name=['RP1', 'Abj_outputs', 'Jhb_outputs', 'RP1_Countries'])

# Extract each DataFrame from the dictionary
df_rp1 = xlsx_file['RP1']
df_abj = xlsx_file['Abj_outputs']
df_jhb = xlsx_file['Jhb_outputs']
df_rp1_countries = xlsx_file['RP1_Countries']

# Define the color map and stage order
color_map = {
    'Contact procedures not initiated': '#f5c6cb',
    '1st or 2nd invites': '#ffeeba',
    '3rd or more invites': '#b8daff',
    'Data sharing discussions and eligibility check': '#d4edda',
    'DTA in progress': '#d6d8db',
    'DTA completed': '#c3e6cb',
    'Data sets in hand': 'green',
    'Database harmonization': '#bee5eb',
    'Ineligible/declined participation/data currently unavailable': '#f5b7b1'
}
stage_order = [
    'Contact procedures not initiated',
    '1st or 2nd invites',
    '3rd or more invites',
    'Data sharing discussions and eligibility check',
    'DTA in progress',
    'DTA completed',
    'Data sets in hand',
    'Database harmonization',
    'Ineligible/declined participation/data currently unavailable'
]

# Convert the month columns in each DataFrame to a datetime format
def map_month_year(df, start_year=2023):
    month_map = {}
    encountered_dec = False  # Flag to indicate if we've encountered 'Dec'
    months_in_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for column in df.columns:
        if any(month in column for month in months_in_order):
            base_month_name = ''.join(filter(str.isalpha, column))
            if base_month_name == 'Jan' and encountered_dec:
                start_year += 1
            if base_month_name == 'Dec':
                encountered_dec = True
            month_map[column] = f'{base_month_name} {start_year}'
    return df.rename(columns=month_map)

df_rp1 = map_month_year(df_rp1)
df_abj = map_month_year(df_abj)
df_jhb = map_month_year(df_jhb)

# Convert column names to datetime
def convert_column_to_datetime(df):
    new_columns = []
    for col in df.columns:
        if col == 'Stage':
            new_columns.append(col)
        else:
            date_str = col.strip() + ' 1'
            new_columns.append(pd.to_datetime(date_str, format='%b %Y %d', errors='coerce'))
    df.columns = new_columns

for dataframe in [df_rp1, df_abj, df_jhb]:
    convert_column_to_datetime(dataframe)

# Function to plot the main stacked bar chart
def plot_stacked_bar_chart(df, title, last_n_months=8, color_map=None, stage_order=None):
    df = df.set_index('Stage').reindex(stage_order).fillna(0).reset_index()
    excluded_df = df[df['Stage'] == 'Ineligible/declined participation/data currently unavailable']
    df = df[df['Stage'] != 'Ineligible/declined participation/data currently unavailable']
    stages_df = df[~df['Stage'].str.contains("Total")]
    transposed_df = stages_df.set_index('Stage').transpose()
    transposed_df = transposed_df.iloc[-last_n_months:]

    fig = px.bar(transposed_df, barmode='stack', title=title,
                 color_discrete_map=color_map)
    fig.update_layout(xaxis_title='Month', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickformat='%b %Y')
    return fig

# Create the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-bar-chart')
        ]), width=12)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='abj-bar-chart')
        ]), width=6),
        dbc.Col(html.Div([
            dcc.Graph(id='jhb-bar-chart')
        ]), width=6)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-countries-bar-chart')
        ]), width=12)
    ])
])

@app.callback(
    [Output('rp1-bar-chart', 'figure'),
     Output('abj-bar-chart', 'figure'),
     Output('jhb-bar-chart', 'figure'),
     Output('rp1-countries-bar-chart', 'figure')],
    [Input('rp1-bar-chart', 'id'),
     Input('abj-bar-chart', 'id'),
     Input('jhb-bar-chart', 'id'),
     Input('rp1-countries-bar-chart', 'id')]
)
def update_charts(*args):
    rp1_fig = plot_stacked_bar_chart(df_rp1, 'RP1 Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    abj_fig = plot_stacked_bar_chart(df_abj, 'Abidjan Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    jhb_fig = plot_stacked_bar_chart(df_jhb, 'Johannesburg Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    
    # Plot the RP1 countries stacked bar chart
    df_rp1_countries.set_index('Study site', inplace=True)
    fig = px.bar(df_rp1_countries, y=stage_order, title='Number of Studies in Relevant African Countries (RP1)',
                 color_discrete_map=color_map, barmode='stack')
    fig.update_layout(xaxis_title='Country', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickangle=-45)
    
    countries_fig = fig
    
    return rp1_fig, abj_fig, jhb_fig, countries_fig

if __name__ == '__main__':
    app.run_server(debug=True)


ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
from jupyter_dash import JupyterDash

app = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Copy the layout and callback code here

if __name__ == '__main__':
    app.run_server(mode='inline')


: 

: 

: 